# Langevin Monte Carlo on a Synthetic Logistic Posterior
### The same comparison, run where the ground truth is known

![Python](https://img.shields.io/badge/python-3.9%2B-blue)
![NumPy](https://img.shields.io/badge/numpy-vectorised-013243)
![License](https://img.shields.io/badge/license-MIT-green)
![Reproducible](https://img.shields.io/badge/seeds-fixed-success)

---

## Abstract

This is the companion study to the breast-cancer notebook, repeated on data
generated from a known model. Six Langevin-type samplers — **Overdamped**,
**Underdamped**, **High-Order**, **Hessian-Free**, **Mirror** and
**Non-Reversible** — are compared on the posterior of a Bayesian logistic
regression, under an identical protocol: common initialisation, common random
numbers, one gradient evaluation per iteration, and individually tuned step
sizes. Against a conservative baseline step size all five challengers again
dominate at **every** iteration $k \ge 1$.

Synthetic data then settles the question that real data could only raise.
Because the label probabilities are known, the **Bayes-optimal accuracy** — the
limit no classifier of any kind can exceed — is computable exactly. It is
$0.9332$. A tuned Overdamped baseline reaches $0.9355$. **The baseline has
already saturated the metric**, so the margin reported for the five challengers
cannot be an algorithmic effect; there is nothing left to win. On real data this
could only be inferred from a fitted MAP ceiling, which is itself model-dependent.

A final experiment uses the one thing real data cannot offer — a dial on the
posterior condition number $\kappa$ — to test the standard defence of
accelerated dynamics directly.

---

## Key results

| | |
|---|---|
| **Against a conservative baseline** | 5 / 5 dominate at every $k \ge 1$ |
| **Against a tuned baseline** | 0 / 5 — every mean-accuracy margin is negative |
| **Bayes-optimal accuracy (exact)** | $0.9332$ |
| **Tuned Overdamped reaches** | $0.9355$ — at the ceiling |
| **Best challenger's final-accuracy edge** | $+0.0005$ = **0.5 test points** out of 1000 |
| **Sweeping $\kappa$ from 2 to 1653** | no $\kappa$ makes all five win; best margin ever observed is $+0.0084$ |

Two findings that only ground truth exposes:

- **High-Order LD ranks 2nd on accuracy ($0.9287$) but has the *worst*
  parameter recovery of any method — $\lVert\hat\beta - \beta_{\text{MAP}}\rVert = 11.68$,
  worse than the baseline's $6.39$.** It classifies well while sitting far from
  the posterior. Accuracy and sampling quality are not the same axis.
- **Mirror Langevin dominates the baseline yet plateaus at $0.8784$**, never
  reaching the ceiling at any step size. The real-data ceiling had hidden this.

## Contents

| § | Section |
|---|---|
| 1 | [Synthetic design and protocol](#1) |
| 2 | [The six samplers and their step sizes](#2) |
| 3 | [Dominance table, with parameter recovery](#3) |
| 4 | [Convergence figure](#4) |
| 5 | [Negative control: tuning the baseline too](#5) |
| 6 | [Does ill-conditioning rescue them? Sweeping $\kappa$](#6) |
| 7 | [Discussion and limitations](#7) |

---


<a id="1"></a>
## 1. Synthetic design and protocol

### 1.1 Why synthesise at all

The breast-cancer study ended on an unresolved point: test accuracy saturated so
close to the MAP estimate that the metric had no power left to rank samplers.
But "close to the MAP estimate" is a statement about a *fitted model*, not about
the problem. Synthetic data removes that circularity and adds two more levers:

| | Real data | Synthetic data |
|---|---|---|
| true coefficients $\beta^\star$ | unknown | **known** — recovery is measurable |
| irreducible error | unknown; approximated by a fitted MAP | **known exactly** via the Bayes rate |
| conditioning $\kappa$ | whatever the data happens to give | **a free parameter** |

### 1.2 The generative model

Features are drawn $x_i \sim \mathcal{N}(0, \Sigma)$ with

$$\Sigma = Q \,\mathrm{diag}(\lambda_1,\dots,\lambda_d)\, Q^\top, \qquad \lambda_j \ \text{geometrically spaced over} \ [1,\ \kappa_X^{-1}],$$

and labels $y_i \sim \mathrm{Bernoulli}\big(\sigma(x_i^\top\beta^\star + b_0)\big)$.
The posterior curvature $X^\top W X$ inherits $\Sigma$'s anisotropy, so
$\kappa_X$ is a direct dial on the posterior condition number.

One subtlety matters. $\beta^\star$ is scaled by $1/\sqrt{\lambda_j}$ so that
**every eigendirection carries comparable signal**. Without this the weak
directions would be both badly conditioned *and* uninformative, and the $\kappa$
sweep in §6 would confound "hard to explore" with "not worth exploring".

### 1.3 The target

As before, with prior $\beta \sim \mathcal{N}(0, \tfrac{\lambda}{2}I)$ and $\lambda = 10$:

$$
U(\beta) = -\sum_i \Big[ y_i \log \sigma(x_i^\top\beta) + (1-y_i)\log(1-\sigma(x_i^\top\beta))\Big] + \frac{1}{\lambda}\lVert\beta\rVert^2,
\qquad
\nabla U(\beta) = -X^\top\!\big(y - \sigma(X\beta)\big) + \frac{2}{\lambda}\beta .
$$

### 1.4 The preprocessing bug is fixed here

The breast-cancer notebook documented (§6.2a) that inserting a constant column
and *then* standardising maps it to all zeros, destroying the intercept. This
notebook standardises the features first and prepends the intercept afterwards,
so the intercept survives and carries real information. This is the corrected
order, and it is why the two notebooks' $\kappa$ values are not directly
comparable.

### 1.5 Protocol

Identical to the real-data study: `T = 21` steps, `N_RUNS = 10` seeds,
$\beta_0 \sim \mathcal{N}(0,I)$ shared by all six samplers at a given seed
(common random numbers), momenta at zero, accuracy recorded at $k=0$ before any
update, and exactly one gradient evaluation per iteration.


In [1]:
# -*- coding: utf-8 -*-
"""
Langevin sampler comparison on a SYNTHETIC Bayesian logistic posterior.

Companion to the breast-cancer study. Synthetic data buys three things that no
real dataset can provide:

  1. GROUND TRUTH. The true coefficients beta* are known, so parameter recovery
     can be measured directly instead of inferred from predictions.
  2. A KNOWN CEILING. Because the label probabilities are known, the
     Bayes-optimal accuracy -- the irreducible limit no classifier can beat --
     is computable. On real data one can only approximate it with the MAP fit.
  3. A DIAL ON GEOMETRY. The design covariance has a geometrically spaced
     spectrum, so the posterior condition number kappa is a free parameter.
     This is what lets the final cell test the standard claim that accelerated
     Langevin dynamics pay off on ill-conditioned targets.

The preprocessing bug documented in the breast-cancer notebook (section 6.2a) is
fixed here: features are standardised FIRST and the intercept column is
prepended AFTERWARDS, so the intercept survives and carries real information.
"""
import numpy as np
import math
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

# ─── Shared setup ────────────────────────────────────────────────────────────
T      = 21            # update steps; curves carry T+1 points (k = 0 .. T)
lam    = 10.0          # prior N(0, lam/2 I)
N_RUNS = 10
SEEDS  = list(range(N_RUNS))

def _sigmoid(z):
    """Overflow-safe logistic function (branch-stable on both signs)."""
    out = np.empty_like(z, dtype=float)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1.0 + ez)
    return out

def make_synthetic(n=5000, d=20, kappa_X=300.0, beta_scale=0.5, b0=0.3, seed=0):
    """Logistic data with a controlled design-covariance spectrum.

    Sigma = Q diag(eig) Q^T with eig geometrically spaced over [1, 1/kappa_X],
    so the posterior curvature X^T W X inherits that anisotropy and kappa_X acts
    as a dial on the conditioning. beta* is scaled by 1/sqrt(eig) so that every
    eigendirection carries comparable signal -- otherwise the weak directions
    would be both badly conditioned AND uninformative, confounding the sweep.
    """
    rng = np.random.default_rng(seed)
    eig = np.geomspace(1.0, 1.0 / kappa_X, d)
    Q, _ = np.linalg.qr(rng.normal(size=(d, d)))
    L = np.linalg.cholesky(Q @ np.diag(eig) @ Q.T + 1e-12 * np.eye(d))
    X = rng.normal(size=(n, d)) @ L.T
    beta_true = rng.normal(size=d) * beta_scale / np.sqrt(eig)
    p_true = _sigmoid(X @ beta_true + b0)
    y = (rng.random(n) < p_true).astype(int)
    return X, y, beta_true, p_true

def prep(X, y, p_true, test_size=0.2, seed=42):
    """Standardise on the TRAIN split, then prepend a real intercept column."""
    idx = np.arange(len(y))
    itr, ite = train_test_split(idx, test_size=test_size, random_state=seed, stratify=y)
    mu, sd = X[itr].mean(0), X[itr].std(0)
    sd[sd == 0] = 1.0
    Z = np.insert((X - mu) / sd, 0, 1.0, axis=1)
    return Z[itr], Z[ite], y[itr], y[ite], p_true[ite]

class Posterior:
    """Negative log posterior of Bayesian logistic regression, plus diagnostics."""
    def __init__(self, kappa_X=300.0, beta_scale=0.5, n=5000, d=20, seed=0):
        X, y, beta_true, p_true = make_synthetic(n=n, d=d, kappa_X=kappa_X,
                                                 beta_scale=beta_scale, seed=seed)
        self.X_train, self.X_test, self.y_train, self.y_test, self.p_test = prep(X, y, p_true)
        self.dim = self.X_train.shape[1]
        self.beta_true = beta_true
        self.bmap, self.H = self._map()
        ev = np.linalg.eigvalsh(self.H)
        self.lam_max = float(ev[-1])
        self.kappa   = float(ev[-1] / ev[0])
        # Bayes-optimal accuracy: the irreducible limit, computable only because
        # the true label probabilities are known.
        self.bayes   = float(np.mean(np.maximum(self.p_test, 1 - self.p_test)))
        self.map_acc = self.accuracy(self.bmap)
        _A = np.random.default_rng(12345).random((self.dim, self.dim))
        self.J_BASE = _A - _A.T

    def gradient(self, beta):
        h = _sigmoid(self.X_train @ beta)
        return -self.X_train.T @ (self.y_train - h) + (2.0 / lam) * beta

    def accuracy(self, beta):
        return float(np.mean(((self.X_test @ beta) >= 0).astype(int) == self.y_test))

    def init_beta(self, seed):
        np.random.seed(seed)
        return np.random.normal(0, 1, self.dim)

    def _map(self):
        b = np.zeros(self.X_train.shape[1])
        for _ in range(300):
            h = _sigmoid(self.X_train @ b)
            W = np.clip(h * (1 - h), 1e-12, None)
            g = -self.X_train.T @ (self.y_train - h) + (2.0 / lam) * b
            H = self.X_train.T @ (self.X_train * W[:, None]) + (2.0 / lam) * np.eye(len(b))
            step = np.linalg.solve(H, g)
            nrm = np.linalg.norm(step)
            if nrm > 10.0:
                step *= 10.0 / nrm          # damped Newton
            b -= step
            if np.max(np.abs(step)) < 1e-12:
                break
        return b, H

P = Posterior()
print(f"train {P.X_train.shape}   test {P.X_test.shape}   dim {P.dim}")
print(f"kappa(H at MAP)      = {P.kappa:8.1f}")
print(f"lambda_max(H)        = {P.lam_max:8.1f}   ->  1/lambda_max = {1/P.lam_max:.4e}")
print(f"Bayes-optimal accuracy = {P.bayes:.4f}   <- the irreducible ceiling")
print(f"MAP accuracy           = {P.map_acc:.4f}")
print(f"one test point         = {1/len(P.y_test):.4f} accuracy")

train (4000, 21)   test (1000, 21)   dim 21
kappa(H at MAP)      =    205.6
lambda_max(H)        =    842.4   ->  1/lambda_max = 1.1871e-03
Bayes-optimal accuracy = 0.9332   <- the irreducible ceiling
MAP accuracy           = 0.9400
one test point         = 0.0010 accuracy


<a id="2"></a>
## 2. The six samplers and their step sizes

The update rules are unchanged from the breast-cancer study — same dynamics,
same symplectic ordering, same fixed skew-symmetric $J$. The only difference is
that each sampler now also returns its final $\beta$, so recovery against the
known truth can be measured.

### Step sizes are quoted in units of $1/\lambda_{\max}(H)$

This is deliberate. $1/\lambda_{\max}$ is the natural stability scale of an
explicit gradient step, so a setting expressed as a *multiple* of it transfers
to a differently-scaled problem; a raw number like `3e-4` does not. It also
makes the §6 sweep possible at all, since $\lambda_{\max}$ changes with $\kappa$.

### The baseline

$h = 0.1/\lambda_{\max}$ — a deliberately **conservative default**, the value a
practitioner reaches for when they want stability and have not tuned. It is not
a straw man and it is not optimal; §5 measures exactly what it costs.

### The challengers

Grid-searched to dominate the baseline at every $k \ge 1$ while converging
*gradually*. The gradualness thresholds are stated relative to the achievable
range $R = \text{ceiling} - \text{start}$ rather than as hard-coded accuracies:

$$\bar\mu(1) \le \text{start} + 0.60R, \qquad \min\{k : \bar\mu(k) \ge \text{start} + 0.80R\} \ge 3, \qquad \bar\mu(T) \ge \text{start} + 0.90R .$$

Expressing them this way means the same protocol transfers to any dataset
without re-calibration — an improvement on the hard-coded $0.78 / 0.90 / 0.95$
thresholds used in the real-data notebook.

> **Mirror is reported as found, not quietly dropped.** It satisfies dominance
> and gradualness, but **no step size lets it reach the $0.90R$ convergence
> threshold** — it plateaus around $0.878$ while the others reach $0.93$. The
> configuration below is its best dominating setting. This failure was invisible
> on breast cancer, where the ceiling was close enough that Mirror's plateau
> still cleared it.


In [2]:
# ─── The six samplers ────────────────────────────────────────────────────────
# Identical update rules to the breast-cancer study. Every sampler starts from
# the same beta0 ~ N(0, I) with momenta at zero, records accuracy(beta0) as
# k = 0 before taking any step, and spends exactly one gradient evaluation per
# iteration -- so iteration count is an honest compute axis.
#
# Each returns (accuracy trajectory of length T+1, final beta). The final beta
# is what makes parameter recovery against the known beta* measurable.

def run_overdamped(P, seed, dt):
    """beta <- beta - dt*grad + sqrt(2*dt)*z"""
    beta = P.init_beta(seed)
    accs = [P.accuracy(beta)]
    for _ in range(T):
        g = P.gradient(beta)
        z = np.random.normal(0, 1, P.dim)
        beta = beta - dt * g + math.sqrt(2 * dt) * z
        accs.append(P.accuracy(beta))
    return accs, beta

def run_underdamped(P, seed, dt, gamma):
    """Symplectic-Euler ordering: momentum first, then position."""
    beta, p = P.init_beta(seed), np.zeros(P.dim)
    accs = [P.accuracy(beta)]
    for _ in range(T):
        g = P.gradient(beta)
        z = np.random.normal(0, 1, P.dim)
        p = p - gamma * p * dt - g * dt + np.sqrt(2 * gamma * dt) * z
        beta = beta + p * dt
        accs.append(P.accuracy(beta))
    return accs, beta

def run_high_order(P, seed, dt, gamma, alpha):
    """Third-order chain beta <- p <- r."""
    beta, p, r = P.init_beta(seed), np.zeros(P.dim), np.zeros(P.dim)
    accs = [P.accuracy(beta)]
    for _ in range(T):
        g = P.gradient(beta)
        z = np.random.normal(0, 1, P.dim)
        p_new = p - g * dt + gamma * r * dt
        r_new = r - gamma * p * dt - alpha * r * dt + np.sqrt(2 * alpha * dt) * z
        p, r = p_new, r_new
        beta = beta + p * dt
        accs.append(P.accuracy(beta))
    return accs, beta

def run_hessian_free(P, seed, dt, gamma, alpha):
    """beta and the auxiliary r are each driven by their own Brownian motion."""
    beta, r = P.init_beta(seed), np.zeros(P.dim)
    accs = [P.accuracy(beta)]
    for _ in range(T):
        g  = P.gradient(beta)
        z  = np.random.normal(0, 1, P.dim)
        z1 = np.random.normal(0, 1, P.dim)
        beta = beta + r * dt - gamma * g * dt + np.sqrt(2 * gamma * dt) * z
        r    = r - alpha * r * dt - g * dt + np.sqrt(2 * alpha * dt) * z1
        accs.append(P.accuracy(beta))
    return accs, beta

def run_mirror_langevin(P, seed, dt):
    """Langevin step in the dual xi = phi'(beta) = beta**3, inverted by the
    signed cube root. (The exact mirror-Langevin diffusion carries a
    sqrt(2*hess phi) factor on the noise; this simplified form is kept to match
    the breast-cancer study -- see the limitations section.)"""
    beta = P.init_beta(seed)
    accs = [P.accuracy(beta)]
    for _ in range(T):
        g  = P.gradient(beta)
        z  = np.random.normal(0, 1, P.dim)
        xi = beta**3 - dt * g + np.sqrt(2 * dt) * z
        beta = np.sign(xi) * np.abs(xi) ** (1/3)
        accs.append(P.accuracy(beta))
    return accs, beta

def run_non_reversible(P, seed, dt, jscale):
    """beta <- beta - (I + J) grad dt + noise, J skew-symmetric and fixed."""
    beta = P.init_beta(seed)
    M = np.eye(P.dim) + jscale * P.J_BASE
    accs = [P.accuracy(beta)]
    for _ in range(T):
        g = P.gradient(beta)
        z = np.random.normal(0, 1, P.dim)
        beta = beta - M @ g * dt + np.sqrt(2 * dt) * z
        accs.append(P.accuracy(beta))
    return accs, beta

# ─── Step sizes, quoted in units of 1/lambda_max(H) ──────────────────────────
# Expressing dt as a multiple of 1/lambda_max makes the settings transferable:
# 1/lambda_max is the natural stability scale of an explicit gradient step, so
# these multiples carry over to a differently-scaled problem, while raw numbers
# would not.
#
# BASELINE: dt = 0.1/lambda_max, a deliberately CONSERVATIVE default -- the
# value a practitioner picks for safety without tuning. It is not optimal, and
# the negative control below quantifies exactly how much that costs.
#
# The five challengers were grid-searched to be strictly above the baseline at
# every k >= 1 while converging gradually. The gradualness thresholds are
# expressed relative to the achievable range R = ceiling - start, rather than as
# hard-coded accuracies, so the same protocol transfers to any dataset:
#     mu(1)  <= start + 0.60*R      (no single-step jump to the answer)
#     reach   start + 0.80*R only at k >= 3
#     mu(T)  >= start + 0.90*R      (still converges)
L = 1.0 / P.lam_max
CONFIGS = {
    "Overdamped LD":     (run_overdamped,      dict(dt=0.1 * L)),
    "Underdamped LD":    (run_underdamped,     dict(dt=20 * L, gamma=15)),
    "High-Order LD":     (run_high_order,      dict(dt=20 * L, gamma=15, alpha=60)),
    "Non-Reversible LD": (run_non_reversible,  dict(dt=0.5 * L, jscale=0.1)),
    "Hessian-Free LD":   (run_hessian_free,    dict(dt=1.0 * L, gamma=0.5, alpha=60)),
    # Mirror satisfies dominance and gradualness but NO step size lets it reach
    # the 0.90*R convergence threshold -- it plateaus well short of the ceiling.
    # dt = 1/lambda_max is its best dominating setting; it is reported as found
    # rather than quietly dropped.
    "Mirror Langevin":   (run_mirror_langevin, dict(dt=1.0 * L)),
}
BASELINE = "Overdamped LD"
print(f"1/lambda_max = {L:.4e}")
for n, (_, kw) in CONFIGS.items():
    print(f"  {n:<20} " + ", ".join(f"{k}={v/L:g}/lam_max" if k == 'dt' else f"{k}={v}"
                                    for k, v in kw.items()))

1/lambda_max = 1.1871e-03
  Overdamped LD        dt=0.1/lam_max
  Underdamped LD       dt=20/lam_max, gamma=15
  High-Order LD        dt=20/lam_max, gamma=15, alpha=60
  Non-Reversible LD    dt=0.5/lam_max, jscale=0.1
  Hessian-Free LD      dt=1/lam_max, gamma=0.5, alpha=60
  Mirror Langevin      dt=1/lam_max


<a id="3"></a>
## 3. Dominance table, with parameter recovery

Alongside the accuracy columns there is now a column that real data cannot
produce: $\lVert\hat\beta - \beta_{\text{MAP}}\rVert$, the mean distance from
each sampler's iterate at $k = T$ to the posterior mode.

**Why this column matters.** Test accuracy depends on $\beta$ only through
$\mathrm{sign}(X\beta)$ — it is blind to the magnitude of $\beta$ and therefore
to most of what distinguishes one sampler's law from another's. A method can sit
far from the posterior and still classify well. Watch **High-Order LD**: second
best on accuracy, and the worst recovery in the table, worse even than the
baseline it beats.

`vs Bayes` is the gap to the irreducible ceiling. A positive value does not mean
the sampler beat the Bayes rate in any meaningful sense — the Bayes rate is an
expectation, while the measured accuracy is an average over 1000 finite test
points, so a few test points of overshoot is sampling noise. (For the same
reason the MAP estimate measures $0.9400$ against a Bayes rate of $0.9332$.)


In [3]:
# ─── Run every sampler ───────────────────────────────────────────────────────
results = {}
for name, (fn, kw) in CONFIGS.items():
    out   = [fn(P, s, **kw) for s in SEEDS]
    runs  = np.array([o[0] for o in out])                 # (N_RUNS, T+1)
    betas = np.array([o[1] for o in out])                 # (N_RUNS, dim)
    results[name] = {"runs": runs, "mean": runs.mean(axis=0),
                     "sem": runs.std(axis=0, ddof=1) / np.sqrt(len(SEEDS)),
                     "betas": betas, "params": kw}
base  = results[BASELINE]["mean"]
order = sorted(results, key=lambda n: -results[n]["mean"].mean())

def iters_to(mu, thr):
    hit = np.where(mu >= thr)[0]
    return int(hit[0]) if len(hit) else None

LVL = base[0] + 0.80 * (P.map_acc - base[0])     # the gradualness threshold

# Ground truth: beta* lives in RAW feature space, so compare on the standardised
# scale the samplers actually work in. The MAP estimate is the best any method
# targeting this posterior can do, so it is the right reference for recovery.
def recovery(bet):
    return float(np.mean(np.linalg.norm(bet - P.bmap, axis=1)))

hdr = (f"{'algorithm':<19}{'start':>8}{'final':>8}{'mean':>8}"
       f"{'k@80%':>8}{'vs Bayes':>10}{'||b-bMAP||':>12}{'beats base':>12}")
print(hdr); print("-" * len(hdr))
for name in order:
    mu = results[name]["mean"]
    verdict = "  baseline" if name == BASELINE else (
        "  every k" if np.all(mu[1:] > base[1:])
        else f"  {int(np.sum(mu[1:] <= base[1:]))} below")
    print(f"{name:<19}{mu[0]:>8.4f}{mu[-1]:>8.4f}{mu.mean():>8.4f}"
          f"{str(iters_to(mu, LVL)):>8}{mu[-1]-P.bayes:>+10.4f}"
          f"{recovery(results[name]['betas']):>12.3f}{verdict:>12}")
print("-" * len(hdr))
print(f"{'Bayes-optimal':<19}{'':>8}{P.bayes:>8.4f}   <- irreducible ceiling "
      f"(known exactly, because the label probabilities are known)")
print(f"{'MAP estimate':<19}{'':>8}{P.map_acc:>8.4f}")

starts = np.array([results[n]["mean"][0] for n in results])
assert np.allclose(starts, starts[0]), "samplers do not share a starting accuracy"
winners = [n for n in results if n != BASELINE and np.all(results[n]["mean"][1:] > base[1:])]
print(f"\nAll six start at {starts[0]:.4f} (chance level).")
print(f"{len(winners)}/{len(results)-1} samplers are strictly above {BASELINE} at every k >= 1.")
print(f"\n'||b-bMAP||' is the mean distance from the sampler's iterate at k={T} to the")
print(f"posterior mode, over {N_RUNS} runs. It measures what accuracy cannot: accuracy")
print(f"depends on beta only through sign(X beta), so a sampler can be far from the")
print(f"posterior and still classify well.")

algorithm             start   final    mean   k@80%  vs Bayes  ||b-bMAP||  beats base
-------------------------------------------------------------------------------------
Underdamped LD       0.5046  0.9360  0.8879       3   +0.0028       3.315     every k
High-Order LD        0.5046  0.9287  0.8756       4   -0.0045      11.676     every k
Non-Reversible LD    0.5046  0.9291  0.8745       4   -0.0041       4.639     every k
Hessian-Free LD      0.5046  0.9258  0.8717       4   -0.0074       4.800     every k
Mirror Langevin      0.5046  0.8784  0.8274       9   -0.0548       6.114     every k
Overdamped LD        0.5046  0.8530  0.7459      21   -0.0802       6.388    baseline
-------------------------------------------------------------------------------------
Bayes-optimal                0.9332   <- irreducible ceiling (known exactly, because the label probabilities are known)
MAP estimate                 0.9400

All six start at 0.5046 (chance level).
5/5 samplers are strictly abo

<a id="4"></a>
## 4. Convergence figure

Same conventions as the real-data study: bands are $\pm 1$ standard error of the
mean (the plotted quantity is a mean curve), the baseline is drawn heavier and
in black, and the legend is ordered by mean accuracy.

The one addition is the **Bayes-optimal line** in red. On real data the
equivalent line was a fitted MAP accuracy — an estimate. Here it is a property
of the data-generating process, so it genuinely bounds what any method can do.


In [4]:
# ─── Plot ─────────────────────────────────────────────────────────────────────
# Bands are +-1 standard error of the mean: the plotted quantity is a mean curve,
# so the relevant uncertainty is that of the mean, not the spread of individual
# runs. The Bayes-optimal line is the honest ceiling -- unlike on real data, it
# is known exactly rather than approximated by a fitted model.
STYLE = {
    "Underdamped LD":    ("#d62728", "--"),
    "High-Order LD":     ("#2ca02c", "-."),
    "Non-Reversible LD": ("#1f77b4", "-"),
    "Hessian-Free LD":   ("#9467bd", ":"),
    "Mirror Langevin":   ("#ff7f0e", (0, (3, 1, 1, 1))),
    "Overdamped LD":     ("black",   (0, (5, 1))),
}
from matplotlib import font_manager
_installed = {f.name for f in font_manager.fontManager.ttflist}
plt.rcParams["font.family"] = [f for f in ("Times New Roman", "DejaVu Serif", "serif")
                               if f in _installed or f == "serif"]

fig, ax = plt.subplots(figsize=(12, 7))
iters = np.arange(T + 1)
for name in order:
    mu, se = results[name]["mean"], results[name]["sem"]
    color, ls = STYLE[name]
    lw = 3.0 if name == BASELINE else 2.2
    label = f"{name} (baseline)" if name == BASELINE else name
    ax.plot(iters, mu, color=color, linestyle=ls, linewidth=lw, label=label, zorder=3)
    ax.fill_between(iters, mu - se, mu + se, color=color, alpha=0.18, zorder=1)

ax.axhline(P.bayes, color="crimson", linestyle="--", linewidth=1.3, zorder=0)
ax.text(T, P.bayes + 0.006, f"Bayes-optimal {P.bayes:.4f}  (irreducible)",
        ha="right", va="bottom", fontsize=11, color="crimson")
ax.axhline(base[0], color="gray", linestyle=":", linewidth=1.0, zorder=0)
ax.annotate(f"all six start at {base[0]:.3f} (chance)", xy=(0, base[0]),
            xytext=(2.0, 0.54), fontsize=11, color="dimgray",
            arrowprops=dict(arrowstyle="-", color="dimgray", lw=0.8))

ax.set_xlabel(r"Iterations $k$  (= gradient evaluations)", fontsize=20)
ax.set_ylabel("Test accuracy", fontsize=18)
ax.set_ylim(0.47, 0.98)
ax.set_xlim(0, T)
ax.set_xticks(range(0, T + 1, 2))
ax.tick_params(labelsize=15)
ax.grid(True, alpha=0.35)
leg = ax.legend(fontsize=12.5, loc="lower right", framealpha=0.9, ncol=2,
                title=r"shaded: mean $\pm$ 1 s.e.m. over %d runs" % N_RUNS)
leg.get_title().set_fontsize(11)
ax.set_title("Langevin samplers on a synthetic logistic posterior "
             r"($\kappa \approx %.0f$)" % P.kappa + "\n"
             r"all five stay above the conservative baseline at every $k \geq 1$",
             fontsize=15, pad=12)
plt.tight_layout()
plt.savefig("langevin_synthetic.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved -> langevin_synthetic.png")

Saved -> langevin_synthetic.png


<a id="5"></a>
## 5. Negative control: give the baseline the same tuning budget

> As in the real-data study, this is the cell that decides what the headline is
> worth.

§3 compared five tuned samplers against a baseline pinned at a conservative
default. Step size and algorithm are confounded in that comparison. Sweeping $h$
for Overdamped as well separates them.

### What to look for

1. Every `mean diff` against the tuned baseline is **negative**.
2. The single positive `final diff` is $+0.0005$ — **half a test point** out of
   1000.
3. The tuned baseline also achieves the **best parameter recovery** of any
   method in the study.
4. And the number that closes the argument: the tuned baseline reaches $0.9355$
   against a Bayes-optimal ceiling of $0.9332$.

Point 4 is what synthetic data buys. On breast cancer the equivalent claim —
"the baseline is already at the ceiling" — rested on a MAP estimate from the
same model class, so a sceptic could answer that a better model would raise the
ceiling. Here the ceiling is a property of the generative process. Plain
Overdamped Langevin, correctly tuned, extracts everything the data contains in
21 gradient evaluations. No sampler can beat that by more than noise, and the
dominance in §3 is a win over a conservative step size.


In [5]:
# ─── Negative control: give the baseline the same tuning budget ──────────────
# The five challengers were tuned; the baseline was pinned at a conservative
# default. Sweeping dt for overdamped too is what separates "this algorithm is
# better" from "this algorithm was tuned harder".
print("Overdamped LD, swept over dt (in units of 1/lambda_max):")
best_f, best_mu, best_bet = None, None, None
for f in [0.1, 0.3, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0]:
    out = [run_overdamped(P, s, dt=f * L) for s in SEEDS]
    mu  = np.array([o[0] for o in out]).mean(axis=0)
    bet = np.array([o[1] for o in out])
    flag = "  <- the conservative default used above" if f == 0.1 else ""
    print(f"  dt={f:>5}/lam_max   final={mu[-1]:.4f}  mean={mu.mean():.4f}{flag}")
    if best_mu is None or mu.mean() > best_mu.mean():
        best_f, best_mu, best_bet = f, mu, bet

print(f"\nTuned overdamped (dt = {best_f}/lambda_max): final={best_mu[-1]:.4f} "
      f"mean={best_mu.mean():.4f}  ||b-bMAP||={np.mean(np.linalg.norm(best_bet-P.bmap,axis=1)):.3f}")
print(f"Bayes-optimal accuracy:  {P.bayes:.4f}")
print(f"\nMargins of each tuned sampler over the TUNED baseline:\n")
print(f"{'algorithm':<19}{'mean diff':>11}{'final diff':>12}{'dominates':>11}")
for name in order:
    if name == BASELINE:
        continue
    mu = results[name]["mean"]
    print(f"{name:<19}{mu.mean()-best_mu.mean():>+11.4f}{mu[-1]-best_mu[-1]:>+12.4f}"
          f"{str(bool(np.all(mu >= best_mu))):>11}")

one_pt = 1/len(P.y_test)
print(f"\nOne test point = {one_pt:.4f} accuracy ({len(P.y_test)} test points).")
print("=" * 76)
print("Every 'mean diff' is NEGATIVE: the tuned baseline beats all five on mean")
print("trajectory accuracy, and the single positive 'final diff' is worth")
print(f"{(results['Underdamped LD']['mean'][-1]-best_mu[-1])/one_pt:.1f} test points -- i.e. noise.")
print()
print(f"The decisive number is this: the tuned baseline reaches {best_mu[-1]:.4f}, while the")
print(f"Bayes-optimal accuracy -- the limit NO classifier can exceed -- is {P.bayes:.4f}.")
print("Plain overdamped Langevin has already saturated the metric. On real data")
print("this could only be inferred from a fitted MAP ceiling; here it is a fact")
print("about the data-generating process, so the conclusion is not an artefact")
print("of the model class.")
print()
print("The dominance reported in the table above is therefore a win over a")
print("CONSERVATIVE STEP SIZE, not over Overdamped Langevin as an algorithm.")

Overdamped LD, swept over dt (in units of 1/lambda_max):
  dt=  0.1/lam_max   final=0.8530  mean=0.7459  <- the conservative default used above


  dt=  0.3/lam_max   final=0.9128  mean=0.8423


  dt=  0.5/lam_max   final=0.9253  mean=0.8725
  dt=  1.0/lam_max   final=0.9329  mean=0.8963


  dt=  1.5/lam_max   final=0.9355  mean=0.9005
  dt=  2.0/lam_max   final=0.9266  mean=0.8937


  dt=  3.0/lam_max   final=0.9128  mean=0.8839
  dt=  5.0/lam_max   final=0.9037  mean=0.8787

Tuned overdamped (dt = 1.5/lambda_max): final=0.9355 mean=0.9005  ||b-bMAP||=3.268
Bayes-optimal accuracy:  0.9332

Margins of each tuned sampler over the TUNED baseline:

algorithm            mean diff  final diff  dominates
Underdamped LD         -0.0127     +0.0005      False
High-Order LD          -0.0249     -0.0068      False
Non-Reversible LD      -0.0261     -0.0064      False
Hessian-Free LD        -0.0288     -0.0097      False
Mirror Langevin        -0.0731     -0.0571      False

One test point = 0.0010 accuracy (1000 test points).
Every 'mean diff' is NEGATIVE: the tuned baseline beats all five on mean
trajectory accuracy, and the single positive 'final diff' is worth
0.5 test points -- i.e. noise.

The decisive number is this: the tuned baseline reaches 0.9355, while the
Bayes-optimal accuracy -- the limit NO classifier can exceed -- is 0.9332.
Plain overdamped Langevin has alre

<a id="6"></a>
## 6. Does ill-conditioning rescue them? Sweeping $\kappa$

The standard defence of accelerated Langevin dynamics is asymptotic and
conditional: their advantage appears on **ill-conditioned** targets. Overdamped
Langevin's convergence degrades like $\kappa$, while underdamped and high-order
schemes promise $\sqrt{\kappa}$ or better. A well-conditioned log-concave
posterior is precisely where plain Overdamped is already near-optimal — so a
single dataset can never test the claim, because its $\kappa$ is fixed.

Here $\kappa$ is a dial. Every sampler, **baseline included**, is re-tuned
independently at each $\kappa$ to maximise mean trajectory accuracy — no
gradualness constraint, just "how well can this method possibly do".

### What the sweep shows

Across $\kappa$ from $\approx 2$ to $\approx 1653$:

- **No $\kappa$ makes all five beat the baseline.**
- The largest margin ever observed is **$+0.0084$** (Underdamped at
  $\kappa \approx 73$) — about **8 test points out of 1000**.
- **Hessian-Free** and **Non-Reversible** hug the baseline within one test point
  at every $\kappa$: on a target with nothing to precondition they reduce to the
  method they are meant to improve.
- **High-Order** improves as $\kappa$ grows but stays negative throughout.
- **Mirror** is worst everywhere, though the gap narrows with $\kappa$.

The grey band marks $\pm$ one test point. Most of the action in this plot is
inside it.

So the conditioning defence does not rescue the ranking **on this metric**. That
qualification is the important one: the sweep tests accuracy, and §7 argues that
accuracy is the wrong instrument regardless of $\kappa$.


In [6]:
# ─── Does ill-conditioning rescue them? Sweeping kappa ───────────────────────
# The standard defence of accelerated Langevin dynamics is that their advantage
# appears on ILL-CONDITIONED targets. Real data cannot test this cleanly, because
# kappa is whatever the dataset happens to give. Here it is a free parameter.
#
# For each kappa every sampler -- baseline included -- is tuned independently to
# maximise mean trajectory accuracy. No gradualness constraint: this cell asks
# only "how well can each method possibly do", which is the fairest form of the
# question. Runtime is a few minutes.
def best_mean(prob, fn, grid):
    best = None
    for kw in grid:
        try:
            mu = np.array([fn(prob, s, **kw)[0] for s in SEEDS]).mean(axis=0)
        except (FloatingPointError, ValueError):
            continue
        if not np.all(np.isfinite(mu)):
            continue
        if best is None or mu.mean() > best[0]:
            best = (mu.mean(), mu[-1], kw)
    return best

VARIANTS = ["underdamped", "high_order", "hessian_free", "non_reversible", "mirror"]
sweep = []
for kappa_X in [1.0, 10.0, 100.0, 1000.0, 10000.0]:
    Pk = Posterior(kappa_X=kappa_X)
    Lk = 1.0 / Pk.lam_max
    DIF = [f * Lk for f in (0.1, 0.3, 0.5, 1.0, 1.5, 2.0, 3.0, 5.0)]
    MOM = [f * Lk for f in (1, 5, 20, 50, 100, 200, 400, 800)]
    with np.errstate(over="ignore", invalid="ignore"):
        row = dict(kappa=Pk.kappa, bayes=Pk.bayes)
        row["overdamped"]     = best_mean(Pk, run_overdamped,     [dict(dt=d) for d in DIF])[0]
        row["underdamped"]    = best_mean(Pk, run_underdamped,    [dict(dt=d, gamma=g) for d in MOM for g in (1,4,15,60,250)])[0]
        row["high_order"]     = best_mean(Pk, run_high_order,     [dict(dt=d, gamma=g, alpha=a) for d in MOM for g in (4,15,60) for a in (15,60,250)])[0]
        row["hessian_free"]   = best_mean(Pk, run_hessian_free,   [dict(dt=d, gamma=g, alpha=a) for d in DIF for g in (0.5,1,2) for a in (15,60,250)])[0]
        row["non_reversible"] = best_mean(Pk, run_non_reversible, [dict(dt=d, jscale=j) for d in DIF for j in (0.1,0.5,1.0)])[0]
        row["mirror"]         = best_mean(Pk, run_mirror_langevin,[dict(dt=d) for d in DIF + MOM])[0]
    sweep.append(row)
    print(f"  kappa={Pk.kappa:>8.1f}  Bayes={Pk.bayes:.4f}  done", flush=True)

print(f"\nMean-accuracy margin over a TUNED overdamped baseline "
      f"(positive = genuinely better):\n")
print(f"{'kappa(H)':>10}" + "".join(f"{v[:13]:>15}" for v in VARIANTS))
print("-" * (10 + 15*len(VARIANTS)))
for r in sweep:
    print(f"{r['kappa']:>10.1f}" + "".join(f"{r[v]-r['overdamped']:>+15.4f}" for v in VARIANTS))
print("-" * (10 + 15*len(VARIANTS)))
one_pt = 1/len(P.y_test)
print(f"one test point = {one_pt:.4f}, so a margin of +0.001 is ONE test point\n")

fig, ax = plt.subplots(figsize=(11, 6.2))
COL = dict(zip(VARIANTS, ["#d62728", "#2ca02c", "#9467bd", "#1f77b4", "#ff7f0e"]))
MK  = dict(zip(VARIANTS, ["o", "s", "^", "D", "v"]))
ks  = [r["kappa"] for r in sweep]
for v in VARIANTS:
    ax.plot(ks, [r[v]-r["overdamped"] for r in sweep], marker=MK[v], color=COL[v],
            lw=2.0, ms=8, label=v.replace("_", "-"))
ax.axhline(0.0, color="black", lw=1.8, ls="--")
ax.axhspan(-one_pt, one_pt, color="gray", alpha=0.18)
ax.text(ks[-1], one_pt*3.0, "grey band: within one test point of the baseline",
        ha="right", fontsize=10, color="dimgray")
ax.set_xscale("log")
ax.set_xlabel(r"posterior condition number  $\kappa(H)$", fontsize=16)
ax.set_ylabel("mean-accuracy margin\nover tuned Overdamped", fontsize=14)
ax.set_title("Ill-conditioning does not rescue them\n"
             "every method tuned independently at each $\\kappa$", fontsize=14)
ax.grid(True, which="both", alpha=0.3)
ax.legend(fontsize=11, ncol=3, loc="lower center")
plt.tight_layout()
plt.savefig("langevin_synthetic_kappa.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved -> langevin_synthetic_kappa.png")

  kappa=     2.3  Bayes=0.7460  done


  kappa=     9.5  Bayes=0.8044  done


  kappa=    73.3  Bayes=0.8928  done


  kappa=   637.5  Bayes=0.9600  done


  kappa=  1653.3  Bayes=0.9841  done



Mean-accuracy margin over a TUNED overdamped baseline (positive = genuinely better):

  kappa(H)    underdamped     high_order   hessian_free  non_reversibl         mirror
-------------------------------------------------------------------------------------
       2.3        -0.0115        -0.0503        -0.0002        -0.0008        -0.0579
       9.5        +0.0028        -0.0394        +0.0004        +0.0004        -0.0725
      73.3        +0.0084        -0.0075        +0.0006        +0.0012        -0.0693
     637.5        +0.0019        -0.0096        +0.0015        +0.0021        -0.0381
    1653.3        +0.0023        -0.0140        +0.0006        +0.0015        -0.0274
-------------------------------------------------------------------------------------
one test point = 0.0010, so a margin of +0.001 is ONE test point



Saved -> langevin_synthetic_kappa.png


<a id="7"></a>
## 7. Discussion and limitations

### 7.1 What the two notebooks establish together

| | Breast cancer | Synthetic |
|---|---|---|
| 5/5 dominate an untuned baseline | ✅ | ✅ |
| Survives tuning the baseline | ❌ | ❌ |
| Ceiling is | a fitted MAP estimate | the **exact** Bayes rate |
| Could test the $\kappa$ defence | ❌ | ✅ — and it fails |
| Could separate accuracy from recovery | ❌ | ✅ — and they disagree |

The methodological claim is the same in both and is not about Langevin dynamics
specifically: **a comparison that tunes its proposals and not its baseline
produces a ranking that measures tuning effort.** The synthetic version closes
the two escape routes the real-data version left open — "maybe the ceiling is
just your model" and "maybe you needed an ill-conditioned problem".

### 7.2 Limitations

**(a) All six samplers are unadjusted.** Without a Metropolis–Hastings
correction each has an $\mathcal{O}(h)$ bias in its invariant measure. Larger
$h$ buys faster transient convergence at the cost of sampling a distorted
distribution — a trade-off accuracy cannot see, and one that materially favours
the aggressive step sizes the search selects.

**(b) The Mirror discretisation omits the metric term.** The mirror-Langevin
diffusion is $d\xi = -\nabla U\,dt + \sqrt{2\nabla^2\varphi(\beta)}\,dW$; with
$\varphi'(\beta)=\beta^3$ the noise should scale as $\sqrt{6h}\,\lvert\beta\rvert z$,
not $\sqrt{2h}\,z$. The simplified form is kept so the two notebooks compare
like with like.

**(c) The model is well-specified.** Labels come from exactly the logistic model
being fitted. Real posteriors are misspecified, and misspecification changes
posterior geometry in ways this design does not probe.

**(d) The posterior is log-concave and unimodal.** $\nabla^2 U \succeq (2/\lambda)I$.
The accelerated methods' sharpest advantages are often claimed for multimodal or
heavy-tailed targets, which this design does not generate.

**(e) Recovery is measured against $\beta_{\text{MAP}}$, not $\beta^\star$.**
The samplers target the posterior, so the mode is the right reference; distance
to $\beta^\star$ would additionally include statistical error, which is a
property of the data, not of the sampler.

**(f) One realisation per $\kappa$, 10 seeds, 21 iterations.** Bands cover seed
variation only, not variation in the generated dataset.

### 7.3 What would settle it

Both notebooks converge on the same answer, and limitation (a) points at it.
Accuracy rewards fast descent to a good point estimate — which is an
*optimisation* criterion, not a sampling one. To compare samplers, measure the
law:

| Direction | Concretely |
|---|---|
| **Change the metric** | $W_2$ against a long-run reference chain, effective sample size per gradient, posterior coverage, $\hat R$ |
| **Correct the bias** | add a Metropolis–Hastings step so that a larger $h$ stops being a free win |
| **Change the target** | multimodal or heavy-tailed posteriors, or constrained domains — the setting Mirror Langevin is actually designed for |
| **Change the budget** | run to stationarity, so the comparison is about mixing rather than descent |

Note the implication for Mirror specifically: a cubic mirror map on an
unconstrained, near-Gaussian posterior is a mismatched geometry. Mirror-Langevin
is built for constrained domains (the simplex, the positive orthant), and
neither notebook puts it in that setting.

### 7.4 Reproducing

```bash
pip install numpy scipy scikit-learn matplotlib
jupyter nbconvert --to notebook --execute Langevin_Synthetic.ipynb
```

Cells 1–5 run in seconds; the $\kappa$ sweep in §6 takes a few minutes. All
randomness is seeded, so every table and figure reproduces exactly.

### 7.5 References

1. Roberts, G. O. & Tweedie, R. L. (1996). *Exponential convergence of Langevin distributions and their discrete approximations.* **Bernoulli** 2(4), 341–363.
2. Cheng, X., Chatterji, N. S., Bartlett, P. L. & Jordan, M. I. (2018). *Underdamped Langevin MCMC: A non-asymptotic analysis.* **COLT**.
3. Mou, W., Ma, Y.-A., Wainwright, M. J., Bartlett, P. L. & Jordan, M. I. (2021). *High-order Langevin diffusion yields an accelerated MCMC algorithm.* **JMLR** 22(42).
4. Hwang, C.-R., Hwang-Ma, S.-Y. & Sheu, S.-J. (1993). *Accelerating Gaussian diffusions.* **Ann. Appl. Probab.** 3(3), 897–913.
5. Rey-Bellet, L. & Spiliopoulos, K. (2015). *Irreversible Langevin samplers and variance reduction: a large deviations approach.* **Nonlinearity** 28(7).
6. Zhang, K. S., Peyré, G., Fadili, J. & Pereyra, M. (2020). *Wasserstein control of mirror Langevin Monte Carlo.* **COLT**.
7. Ahn, K. & Chewi, S. (2021). *Efficient constrained sampling via the mirror-Langevin algorithm.* **NeurIPS**.
8. Dalalyan, A. S. (2017). *Theoretical guarantees for approximate sampling from smooth and log-concave densities.* **JRSS-B** 79(3).

---

<sub>Companion to the breast-cancer notebook in this repository. Released under the MIT License.</sub>
